In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm
from matplotlib import pyplot as plt
import plotly.graph_objects as go

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)

In [2]:
# ==========================================
# Your existing data
# X should have shape (n_samples, 2)
# y should have shape (n_samples,)
# ==========================================

# Example
X = np.load('function_2/initial_inputs.npy')
y = np.load('function_2/initial_outputs.npy')

X = np.vstack([X, [0.695196, 0.039822],
               [0.842003, 0.926535],
               [0.684540, 0.926564],
               	[0.729973, 0.926399],
                [0.707457, 0.926452],
                [0.759337, 0.029195],
                [0.702329, 0.926564],
                [0.691003, 0.926335],
                [0.692599, 0.925979],
                [0.690655, 0.926543],
                [0.689456, 0.926072]])

y = np.append(y, [0.48793657632800824,
                  0.1426898742067465,
                  0.5368989937071333,
                  0.5256790048837894,
                  0.5579919041960164,
                  0.3334878049101354,
                  0.4514686561475304,
                  0.5385616052535137,
                  0.6457198424507699,
                  0.6298616566297821,
                  0.6593395398963507])



# Create the 3D scatter plot
# Plot all points except the last one
# Create figure
fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=X[:-1, 0],
        y=X[:-1, 1],
        z=y[:-1],
        mode="markers",
        marker=dict(
            size=4,
            color="blue",
            opacity=0.8,
        ),
        name="Data points",
    )
)

# Plot the last point with a different color
fig.add_trace(
    go.Scatter3d(
        x=[X[-1, 0]],
        y=[X[-1, 1]],
        z=[y[-1]],
        mode="markers",
        marker=dict(
            size=8,
            color="red",
            symbol="diamond",
        ),
        name="Last point",
    )
)

# Layout
fig.update_layout(
    title="3D Scatter Plot with Last Point Highlighted",
    scene=dict(
        xaxis_title="X1",
        yaxis_title="X2",
        zaxis_title="y",
    ),
    template="plotly_white",
)

fig.show()

In [3]:
# ==========================================
# Fit Gaussian Process
# ==========================================

kernel = (
    ConstantKernel(1.0)
    * Matern(length_scale=[1.0, 1.0], nu=2.5)
    + WhiteKernel(noise_level=1e-6)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=0
)

gp.fit(X, y)

# 3. Create a dense grid to map the continuous 3D surface
bounds = list(zip(X.min(axis=0), X.max(axis=0))) # replace with your limits
bounds = np.array(bounds)
x_line = np.linspace(bounds[0, 0], bounds[0, 1], 100)
y_line = np.linspace(bounds[1, 0], bounds[1, 1], 100)
X_grid, Y_grid = np.meshgrid(x_line, y_line)

grid_points = np.vstack([X_grid.ravel(), Y_grid.ravel()]).T

# Predict scores across the entire dense grid
Z_grid = gp.predict(grid_points).reshape(X_grid.shape)

# 4. Define the Classification Threshold (e.g., top 20% of outputs are "Good")
threshold = np.percentile(y, 80)
# Create a flat 3D plane reflecting our decision boundary
Z_boundary = np.full_like(Z_grid, fill_value=threshold)

# 5. Construct the Interactive Plotly 3D Figure
fig = go.Figure()

# Trace 1: The Predicted Continuous Response Surface
fig.add_trace(go.Surface(
    x=x_line, y=y_line, z=Z_grid,
    colorscale='Viridis',
    name='Predicted Landscape',
    colorbar=dict(title='Score Value')
))

# Trace 2: The Semi-Transparent Decision Boundary Plane
fig.add_trace(go.Surface(
    x=x_line, y=y_line, z=Z_boundary,
    colorscale='Reds',
    opacity=0.4,
    showscale=False,
    name='Good/Bad Threshold'
))

# Trace 3: Real Evaluated Points (Scatter)
fig.add_trace(go.Scatter3d(
    x=X[:, 0], y=X[:, 1], z=y,
    mode='markers',
    marker=dict(size=5, color='orange', symbol='circle'),
    name='Submitted Queries'
))

# Layout configurations for crisp visualization
fig.update_layout(
    title='BBO 3D Surface Plot with Classification Decision Boundary',
    scene=dict(
        xaxis_title='Feature 1',
        yaxis_title='Feature 2',
        zaxis_title='Output Score'
    ),
    margin=dict(l=0, r=0, b=0, t=50)
)

# Display the interactive window
fig.show()




In [4]:
# ==========================================
# Fit Gaussian Process
# ==========================================

kernel = (
    ConstantKernel(1.0)
    * Matern(length_scale=[1.0, 1.0], nu=2.5)
    + WhiteKernel(noise_level=1e-6)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=0
)

gp.fit(X, y)

# ==========================================
# Expected Improvement
# ==========================================

def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)

    mu, sigma = gp.predict(x, return_std=True)

    sigma = sigma[0]
    mu = mu[0]

    if sigma < 1e-12:
        return 0.0

    improvement = mu - y_best - xi
    Z = improvement / sigma

    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)

    return ei



# ==========================================
# Optimise acquisition function
# ==========================================

bounds = list(zip(X.min(axis=0), X.max(axis=0))) # replace with your limits

y_best = np.max(y)


def objective(x):
    return -expected_improvement(x, gp, y_best)


best_x = None
best_ei = -np.inf

for _ in range(10000):

    x0 = np.random.uniform(
        [b[0] for b in bounds],
        [b[1] for b in bounds]
    )

    res = minimize(
        objective,
        x0=x0,
        bounds=bounds,
        method="L-BFGS-B"
    )

    ei = -res.fun

    if ei > best_ei:
        best_ei = ei
        best_x = res.x

print("Next point to evaluate:")
next_point = np.round(best_x, decimals=6)
print(next_point)

print("Expected Improvement:")
print(best_ei)

Next point to evaluate:
[0.685234 0.926564]
Expected Improvement:
0.0032032304784229016
